In [5]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
from scipy.signal import welch
import mne
import google.generativeai as genai
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableMap
import os 
from dotenv import load_dotenv

# -------------------------
# Gemini Config
# -------------------------

load_dotenv()
API_KEY = os.getenv("API_KEY")
genai.configure(api_key=API_KEY)  # Replace with your  API key
model = genai.GenerativeModel("gemini-2.0-flash-exp")

# -------------------------
# Prompt Template
# -------------------------

prompt = PromptTemplate.from_template(
    """
You are a board-certified clinical neurophysiologist. Based on the following quantitative EEG summary, generate a structured EEG report according to ACNS Guideline 7. Limit your response to the three main interpretive sections outlined below.

Each section should use precise and objective language suitable for an official medical report. Do not speculate or provide educational explanations.

1. EEG Description
Describe the background activity including rhythm type, frequency, symmetry, amplitude, and reactivity.  
Include information about slowing (focal/generalized), organization, and presence of epileptiform activity (spikes, sharp waves), noting their morphology, distribution, and frequency.  
Also include relevant spectral power findings and statistical abnormalities (e.g., high kurtosis, skewness, variance) and any artifacts if present.

2. Impression  
Summarize the EEG findings clearly and concisely, emphasizing the dominant rhythm, significant abnormalities, and whether the findings are normal or abnormal.  
If abnormalities are present, describe their likely anatomical distribution.  
Avoid definitive diagnoses (e.g., "epilepsy"); instead, describe observed patterns consistent with particular types of dysfunction (e.g., "intermittent temporal sharp waves consistent with focal irritative zone").

3. Clinical Correlation 
Indicate the potential clinical relevance of the findings, emphasizing that interpretation should be made in the context of the patient's history and clinical presentation.  
If the EEG is within normal limits for age, state this explicitly. If abnormal, suggest that the findings may warrant further evaluation depending on clinical context.

- Limit the report to approximately 2000 characters. Do not repeat content.

{summary}

Report:
"""
)

# -------------------------
# EEG Summary Function
# -------------------------

RELEVANT_CHANNELS = [
    'EEG FP1-REF', 'EEG FP2-REF', 'EEG F3-REF', 'EEG F4-REF',
    'EEG C3-REF', 'EEG C4-REF', 'EEG P3-REF', 'EEG P4-REF',
    'EEG O1-REF', 'EEG O2-REF', 'EEG F7-REF', 'EEG F8-REF',
    'EEG T3-REF', 'EEG T4-REF', 'EEG T5-REF', 'EEG T6-REF',
    'EEG CZ-REF'
]

BANDS = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 12),
    'beta': (12, 30)
}

REGION_MAP = {
    "F": "frontal", "T": "temporal", "P": "parietal",
    "O": "occipital", "C": "central", "Z": "midline"
}

def bandpower(signal, sf, band):
    freqs, psd = welch(signal, sf, nperseg=min(256, len(signal)))
    idx = np.logical_and(freqs >= band[0], freqs <= band[1])
    return np.trapz(psd[idx], freqs[idx]) / np.trapz(psd, freqs)

def peak_frequency(signal, sf):
    freqs, psd = welch(signal, sf, nperseg=min(256, len(signal)))
    return freqs[np.argmax(psd)]

def detect_spikes(signal, kurt=None, delta_power=None):
    base_thresh = 3 * np.std(signal)
    if kurt is not None and delta_power is not None:
        if kurt > 5 or delta_power > 0.2:
            base_thresh *= 0.8
    return int(np.any(np.abs(signal) > base_thresh))

def summarize_eeg_session(edf_path, metadata=None, segment_length=5, sfreq=250, max_duration=90):
    try:
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        if raw.times[-1] > max_duration:
            print(f"Truncating {edf_path} to {max_duration} seconds")
        raw.crop(tmax=max_duration)
        raw.resample(sfreq)
        raw.notch_filter(freqs=60)
        raw.filter(0.5, 45)

        raw.pick([ch for ch in RELEVANT_CHANNELS if ch in raw.ch_names])
        if len(raw.ch_names) < 5:
            return None

        epochs = mne.make_fixed_length_epochs(raw, duration=segment_length, preload=True, verbose=False)
        data = epochs.get_data()

        band_summaries = {band: [] for band in BANDS}
        spike_regions = {}
        peak_freqs = []
        stats = []
        spike_by_channel = {}

        for epoch in data:
            for i, ch_signal in enumerate(epoch):
                ch_name = raw.ch_names[i]
                region = "unknown"
                for key, label in REGION_MAP.items():
                    if key in ch_name:
                        region = label
                        break

                for band, freq_range in BANDS.items():
                    band_summaries[band].append(bandpower(ch_signal, sfreq, freq_range))

                peak_freqs.append(peak_frequency(ch_signal, sfreq))
                stats.append({
                    "var": np.var(ch_signal),
                    "skew": skew(ch_signal),
                    "kurtosis": kurtosis(ch_signal)
                })

                delta_power = bandpower(ch_signal, sfreq, BANDS['delta'])
                kurt_val = stats[-1]['kurtosis']
                if detect_spikes(ch_signal, kurt=kurt_val, delta_power=delta_power):
                    spike_regions[region] = spike_regions.get(region, 0) + 1
                    spike_by_channel[ch_name] = spike_by_channel.get(ch_name, 0) + 1

        band_summary_text = "\n".join(
            f"- {band.capitalize()} Band Power: {np.mean(vals):.2f} ± {np.std(vals):.2f}" for band, vals in band_summaries.items()
        )

        avg_stats = {
            "variance": np.mean([s["var"] for s in stats]),
            "skewness": np.mean([s["skew"] for s in stats]),
            "kurtosis": np.mean([s["kurtosis"] for s in stats])
        }

        summary = (
            f"Patient ID: {metadata.get('subject_id', 'Unknown')}\n"
            f"Age: {metadata.get('age', 'Unknown')}\n"
            f"Sex: {metadata.get('gender', 'Unknown')}\n"
            f"EEG Summary:\n"
            f"{band_summary_text}\n"
            f"- Peak Frequency Avg: {np.mean(peak_freqs):.2f} Hz\n"
            f"- Variance: {avg_stats['variance']:.2f}, Skewness: {avg_stats['skewness']:.2f}, Kurtosis: {avg_stats['kurtosis']:.2f}\n"
            f"- Spikes Detected In Regions: {', '.join([f'{k} ({v})' for k, v in spike_regions.items()]) or 'None'}\n"
            f"- Spike Channels: {', '.join([f'{ch}: {cnt}' for ch, cnt in spike_by_channel.items()]) or 'None'}\n"
            f"Recording duration: {segment_length * len(data)} seconds."
        )
        return summary
    except Exception as e:
        print("Error summarizing EEG:", e)
        return None

# -----------------------------
# Chain: EDF → Summary → Report
# -----------------------------

def build_chain():
    return (
        RunnableMap({
            "summary": lambda x: summarize_eeg_session(x["edf_path"], x["metadata"])
        })
        | RunnableMap({
            "prompt": lambda x: prompt.format(summary=x["summary"])
        })
        | RunnableMap({
        "full_output": lambda x: model.generate_content(
            x["prompt"],
            generation_config={
            "temperature": 0.4,            
            "max_output_tokens": 1000      
        }).text
        })
        | RunnableMap({
            "report_only": lambda x: x["full_output"].split("Report:", 1)[-1].strip()
        })
        
    )

In [6]:
import traceback
import pandas as pd

if __name__ == "__main__":
    # Load metadata and filter by montage
    metadata_df = pd.read_excel("resources/eeg_metadata.xlsx")
    metadata_df = metadata_df[metadata_df["montage"] == "01_tcp_ar"]
    metadata_df["epilepsy"] = metadata_df["patient_group"].map({"epilepsy": 1, "no_epilepsy": 0})

    # Balance sample: 3 from each group (adjust as needed)
    n_per_group = 3
    epileptic_df = metadata_df[metadata_df["epilepsy"] == 1].sample(n=n_per_group, random_state=42)
    control_df = metadata_df[metadata_df["epilepsy"] == 0].sample(n=n_per_group, random_state=42)
    sample_df = pd.concat([epileptic_df, control_df]).sample(frac=1, random_state=42)  # shuffle

    chain = build_chain()
    reports = []

    for i, sample in sample_df.iterrows():
        edf_path = sample["edf_path"]
        metadata = {
            "subject_id": sample["subject_id"],
            "age": sample["age"],
            "gender": sample["gender"],
            "epilepsy": sample["epilepsy"]
        }

        try:
            result = chain.invoke({"edf_path": edf_path, "metadata": metadata})
            report_text = result["report_only"]
        except Exception as e:
            traceback.print_exc()
            print(f"Failed to generate report for {edf_path}: {e}")
            report_text = "Generation failed."

        reports.append({
            "subject_id": metadata["subject_id"],
            "age": metadata["age"],
            "gender": metadata["gender"],
            "epilepsy": metadata["epilepsy"],
            "edf_path": edf_path,
            "generated_report": report_text
        })

    reports_df = pd.DataFrame(reports)
    reports_df.to_excel("eeg_generated_reports_balanced.xlsx", index=False)


Truncating data/00_epilepsy/aaaaalib/s005_2012/01_tcp_ar/aaaaalib_s005_t005.edf to 90 seconds
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 1651 samples (6.604 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


Truncating data/00_epilepsy/aaaaaovm/s004_2013/01_tcp_ar/aaaaaovm_s004_t010.edf to 90 seconds
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 1651 samples (6.604 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


Error summarizing EEG: tmax (90) must be less than or equal to the max time (0.9961 s)
Truncating data/00_epilepsy/aaaaaicb/s001_2011/01_tcp_ar/aaaaaicb_s001_t000.edf to 90 seconds
Sampling frequency of the instance is already 250.0, returning unmodified.
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 1651 samples (6.604 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal ba

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Error summarizing EEG: tmax (90) must be less than or equal to the max time (78.9960 s)
Error summarizing EEG: tmax (90) must be less than or equal to the max time (50.9960 s)
